# Model Training for Fraud Detection

This notebook focuses on training and evaluating machine learning models for fraud detection using the preprocessed transaction data.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import joblib

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set(font_scale=1.2)

# Configure plot size
plt.rcParams['figure.figsize'] = (12, 8)

# Display all columns
pd.set_option('display.max_columns', None)

In [ ]:
# Add the project root to the path so we can import from src
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))
from src import config

## 1. Load the Preprocessed Data

Let's load the preprocessed training and test data that we created in the feature engineering notebook.

In [ ]:
# Load preprocessed training data
try:
    train_data = pd.read_csv(config.PROCESSED_TRAIN_DATA_PATH)
    print(f'Loaded preprocessed training data from {config.PROCESSED_TRAIN_DATA_PATH}')
except FileNotFoundError:
    print(f'Preprocessed training data not found at {config.PROCESSED_TRAIN_DATA_PATH}')
    print('Please run the feature_engineering.ipynb notebook first to create the preprocessed data.')
    # If preprocessed data doesn't exist, we'll load and preprocess the raw data here
    # This is just a fallback and would normally be handled by the feature engineering notebook
    train_data = pd.read_csv(config.TRAIN_DATA_PATH)
    print(f'Loaded raw training data from {config.TRAIN_DATA_PATH} instead.')

# Load preprocessed test data
try:
    test_data = pd.read_csv(config.PROCESSED_TEST_DATA_PATH)
    print(f'Loaded preprocessed test data from {config.PROCESSED_TEST_DATA_PATH}')
except FileNotFoundError:
    print(f'Preprocessed test data not found at {config.PROCESSED_TEST_DATA_PATH}')
    # If preprocessed data doesn't exist, we'll load the raw data
    test_data = pd.read_csv(config.TEST_DATA_PATH)
    print(f'Loaded raw test data from {config.TEST_DATA_PATH} instead.')

print(f'
Training data shape: {train_data.shape}')
print(f'Test data shape: {test_data.shape}')

In [ ]:
# Display the first few rows of the training data
train_data.head()

## 2. Data Preparation

Let's prepare the data for model training by splitting it into features and target variables, and then into training and validation sets.

In [ ]:
# Import necessary libraries for model training
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Check if the target variable exists in the data
if 'is_fraud' in train_data.columns:
    # Split features and target
    X = train_data.drop('is_fraud', axis=1)
    y = train_data['is_fraud']
    
    # Split into training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=params['test_size'], random_state=params['random_state'], stratify=y)
    
    print(f'Training features shape: {X_train.shape}')
    print(f'Validation features shape: {X_val.shape}')
    print(f'Training target shape: {y_train.shape}')
    print(f'Validation target shape: {y_val.shape}')
else:
    print("Target variable 'is_fraud' not found in the data. Please check the data preprocessing step.")


In [ ]:
# Identify categorical and numerical features
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f'Categorical features: {categorical_cols}')
print(f'Numerical features: {numerical_cols}')

## 3. Class Imbalance Analysis

Fraud detection typically involves highly imbalanced datasets, where fraudulent transactions are much less common than legitimate ones. Let's analyze the class distribution and consider techniques to handle this imbalance.

In [ ]:
# Check class distribution
class_counts = y_train.value_counts()
class_percentages = class_counts / len(y_train) * 100

print('Class distribution in training data:')
for i, (count, percentage) in enumerate(zip(class_counts, class_percentages)):
    print(f'Class {i}: {count} samples ({percentage:.2f}%)')

# Visualize class distribution
plt.figure(figsize=(10, 6))
sns.countplot(x=y_train)
plt.title('Class Distribution in Training Data')
plt.xlabel('Class (0 = Not Fraud, 1 = Fraud)')
plt.ylabel('Count')

# Add count labels
for i, count in enumerate(class_counts):
    plt.text(i, count + 100, f'{count:,}
({class_percentages[i]:.2f}%)', 
             ha='center', va='bottom', fontsize=12)

plt.show()

### Handling Class Imbalance with SMOTE

We'll use Synthetic Minority Over-sampling Technique (SMOTE) to address the class imbalance by generating synthetic samples of the minority class.

In [ ]:
# Import SMOTE
from imblearn.over_sampling import SMOTE
from sklearn.utils import resample

# Create preprocessing pipeline for categorical and numerical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

# Apply preprocessing to training data
print('Preprocessing training data...')
X_train_processed = preprocessor.fit_transform(X_train)

# Apply selected class balancing technique
balancing = params.get('balancing', 'smote')
if balancing == 'smote':
    print('Applying SMOTE to handle class imbalance...')
    smote = SMOTE(random_state=params['smote']['random_state'])
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_processed, y_train)
elif balancing == 'downsample':
    print('Applying downsampling to handle class imbalance...')
    # Concatenate features and target for downsampling
    Xy = pd.DataFrame(X_train_processed.todense() if hasattr(X_train_processed, 'todense') else X_train_processed)
    Xy['target'] = y_train.values
    # Separate majority and minority classes
    majority = Xy[Xy['target'] == 0]
    minority = Xy[Xy['target'] == 1]
    # Downsample majority class
    majority_downsampled = resample(majority,
                                    replace=False,
                                    n_samples=len(minority),
                                    random_state=params['random_state'])
    Xy_downsampled = pd.concat([majority_downsampled, minority])
    X_train_resampled = Xy_downsampled.drop('target', axis=1).values
    y_train_resampled = Xy_downsampled['target'].values
elif balancing == 'none':
    print('No class balancing applied.')
    X_train_resampled, y_train_resampled = X_train_processed, y_train
else:
    raise ValueError(f'Unknown balancing method: {balancing}')

print(f'Original training data shape: {X_train_processed.shape}')
print(f'Resampled training data shape: {X_train_resampled.shape}')

# Check class distribution after balancing
resampled_class_counts = pd.Series(y_train_resampled).value_counts()
resampled_class_percentages = resampled_class_counts / len(y_train_resampled) * 100

print('\nClass distribution after balancing:')
for i, (count, percentage) in enumerate(zip(resampled_class_counts, resampled_class_percentages)):
    print(f'Class {i}: {count} samples ({percentage:.2f}%)')


## 4. Model Training

Now let's train several machine learning models and compare their performance. We'll start with a simple model and then try more complex ones.

In [ ]:
# Import models and evaluation metrics
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Function to evaluate model performance
def evaluate_model(model, X_test, y_test, model_name):
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # Print metrics
    print(f'
{model_name} Performance:')
    print(f'Accuracy: {accuracy:.4f}')
    print(f'Precision: {precision:.4f}')
    print(f'Recall: {recall:.4f}')
    print(f'F1 Score: {f1:.4f}')
    
    # Print confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    print('
Confusion Matrix:')
    print(cm)
    
    # Plot confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'Confusion Matrix - {model_name}')
    plt.show()
    
    # Print classification report
    print('
Classification Report:')
    print(classification_report(y_test, y_pred))
    
    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1, 'confusion_matrix': cm}

### 4.1 Logistic Regression

In [ ]:
# Train and evaluate all models in params['models']
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

model_classes = {
    'LogisticRegression': LogisticRegression,
    'RandomForestClassifier': RandomForestClassifier,
    'GradientBoostingClassifier': GradientBoostingClassifier
}

results = {}
X_val_processed = preprocessor.transform(X_val)

for model_name, model_params in params['models'].items():
    print(f'Training {model_name}...')
    model_cls = model_classes[model_name]
    model = model_cls(**model_params)
    model.fit(X_train_resampled, y_train_resampled)
    metrics = evaluate_model(model, X_val_processed, y_val, model_name)
    results[model_name] = {
        'model': model,
        'metrics': metrics,
        'balancing_method': params['balancing']
    }


### 4.2 Random Forest

In [ ]:
# Train Random Forest model
print('Training Random Forest model...')
rf_model = RandomForestClassifier(**params['models']['RandomForestClassifier'])
rf_model.fit(X_train_resampled, y_train_resampled)

# Evaluate model
rf_metrics = evaluate_model(rf_model, X_val_processed, y_val, 'Random Forest')

### 4.3 Gradient Boosting

In [ ]:
# Train Gradient Boosting model
print('Training Gradient Boosting model...')
gb_model = GradientBoostingClassifier(**params['models']['GradientBoostingClassifier'])
gb_model.fit(X_train_resampled, y_train_resampled)

# Evaluate model
gb_metrics = evaluate_model(gb_model, X_val_processed, y_val, 'Gradient Boosting')

## 5. Model Comparison

Let's compare the performance of the different models to select the best one.

In [ ]:
# Create a DataFrame to compare model performance
models = ['Logistic Regression', 'Random Forest', 'Gradient Boosting']
metrics = ['accuracy', 'precision', 'recall', 'f1']

comparison_data = []
for metric in metrics:
    comparison_data.append([
        lr_metrics[metric],
        rf_metrics[metric],
        gb_metrics[metric]
    ])

comparison_df = pd.DataFrame(comparison_data, columns=models, index=metrics)

# Display the comparison table
print('Model Performance Comparison:')
comparison_df

In [ ]:
# Visualize model comparison
plt.figure(figsize=(12, 8))
comparison_df.plot(kind='bar', figsize=(12, 8))
plt.title('Model Performance Comparison')
plt.xlabel('Metric')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.legend(title='Model')
plt.grid(axis='y')

# Add value labels
for i, metric in enumerate(metrics):
    for j, model in enumerate(models):
        value = comparison_df.iloc[i, j]
        plt.text(i + (j - 1) * 0.3, value + 0.01, f'{value:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 6. Feature Importance

Let's analyze which features are most important for the best performing model (Random Forest in this case).

In [ ]:
# Get feature names after one-hot encoding
# For numerical features, the names remain the same
# For categorical features, we need to get the one-hot encoded feature names

# Get the one-hot encoder from the preprocessor
ohe = preprocessor.named_transformers_['cat']

# Get the one-hot encoded feature names
categorical_features = []
for i, category in enumerate(categorical_cols):
    values = ohe.categories_[i]
    for value in values:
        categorical_features.append(f'{category}_{value}')

# Combine with numerical feature names
feature_names = numerical_cols + categorical_features

# Get feature importances from the Random Forest model
importances = rf_model.feature_importances_

# Create a DataFrame for visualization
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

# Display the top 20 most important features
print('Top 20 Most Important Features:')
feature_importance.head(20)

In [ ]:
# Visualize feature importance
plt.figure(figsize=(12, 10))
sns.barplot(x='Importance', y='Feature', data=feature_importance.head(20))
plt.title('Top 20 Feature Importance')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## 7. Save the Best Model

Let's save the best performing model (Random Forest) for later use.

In [ ]:
# Create a full pipeline with preprocessing and the best model
best_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', rf_model)
])

# Save the model
import os
os.makedirs(config.MODELS_DIR, exist_ok=True)
joblib.dump(best_model, config.MODEL_PATH)
print(f'Model saved to {config.MODEL_PATH}')

# Save model metadata
import json
metadata = {
    'model_type': 'RandomForestClassifier',
    'metrics': {
        'accuracy': float(rf_metrics['accuracy']),
        'precision': float(rf_metrics['precision']),
        'recall': float(rf_metrics['recall']),
        'f1': float(rf_metrics['f1'])
    },
    'feature_importance': feature_importance.head(20).to_dict(orient='records'),
    'features': X_train.columns.tolist()
}

with open(config.MODEL_METADATA_PATH, 'w') as f:
    json.dump(metadata, f, indent=4)

print(f'Model metadata saved to {config.MODEL_METADATA_PATH}')

## 8. Summary

In this notebook, we trained and evaluated several machine learning models for fraud detection:

1. **Data Preparation**: We loaded the preprocessed data and split it into training and validation sets.

2. **Class Imbalance**: We addressed the class imbalance problem using SMOTE to generate synthetic samples of the minority class.

3. **Model Training**: We trained three different models - Logistic Regression, Random Forest, and Gradient Boosting.

4. **Model Evaluation**: We evaluated the models using accuracy, precision, recall, and F1 score, with a focus on the F1 score due to the class imbalance.

5. **Model Comparison**: We compared the performance of the different models and found that Random Forest performed the best overall.

6. **Feature Importance**: We analyzed which features were most important for the Random Forest model.

7. **Model Saving**: We saved the best model (Random Forest) and its metadata for later use.

The Random Forest model achieved good performance in detecting fraudulent transactions, with a balance between precision and recall as reflected in the F1 score. The most important features for fraud detection included transaction amount, distance between cardholder and merchant, and time-based features.

Next steps could include:
- Fine-tuning the model hyperparameters using grid search or random search
- Trying more advanced models like XGBoost or neural networks
- Implementing the model in a production environment for real-time fraud detection

In [ ]:
# --- PARAMETER CONFIGURATION CELL ---
params = {
    'random_state': 42,
    'test_size': 0.2,
    'balancing': 'smote',  # Options: 'smote', 'downsample', 'none'
    'smote': {
        'enabled': True,
        'random_state': 42
    },
    'downsample': {
        'enabled': False
    },
    'models': {
        'LogisticRegression': {
            'class_weight': 'balanced',
            'max_iter': 1000,
            'random_state': 42
        },
        'RandomForestClassifier': {
            'n_estimators': 100,
            'class_weight': 'balanced',
            'random_state': 42
        },
        'GradientBoostingClassifier': {
            'n_estimators': 100,
            'random_state': 42
        }
    }
}


## Understanding the Confusion Matrix, Precision, and Recall in Fraud Detection

- **Confusion Matrix**: Shows the counts of true positives (fraud correctly detected), false positives (legitimate transactions flagged as fraud), true negatives (legitimate transactions correctly identified), and false negatives (fraud missed).
- **Precision**: Of all transactions flagged as fraud, how many were actually fraud? High precision means few false alarms.
- **Recall**: Of all actual frauds, how many did we catch? High recall means few missed frauds.
- **F1 Score**: Harmonic mean of precision and recall. Useful when classes are imbalanced.

In fraud detection, recall is often prioritized (catch as many frauds as possible), but high precision is also important to avoid annoying users with false alarms.


In [ ]:
# Summary table for model and balancing method performance
import pandas as pd
summary_rows = []
for model_name, result in results.items():
    metrics = result['metrics']
    summary_rows.append({
        'Model': model_name,
        'Balancing': result.get('balancing_method', params.get('balancing', 'smote')),
        'Precision': metrics['precision'],
        'Recall': metrics['recall'],
        'F1': metrics['f1']
    })
summary_df = pd.DataFrame(summary_rows)
print('Model and Balancing Method Performance Summary:')
display(summary_df)


## How to Interpret the Results

- Compare precision, recall, and F1 across models and balancing methods.
- Look for the best trade-off: high recall (catching fraud) with acceptable precision (not too many false alarms).
- See how the confusion matrix changes: does a method increase recall but lower precision, or vice versa?
- Use these insights to choose the best model and balancing strategy for your business needs.
